# Task 1.1 Data Loading & Setup

In [5]:
import os
import glob
import pandas as pd
import ast
import json
import random
from tqdm import tqdm

DATA_DIR = "../../data/datasets/combined_dataset"
RECIPE_CSV = "../../data/datasets/recipe_dataset_2m.csv"

# 1. Parse combined_dataset folder
categories = [d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d))]
print(f"Total categories found: {len(categories)}")

# Collect all image paths
all_images = []
for cat in categories:
    cat_dir = os.path.join(DATA_DIR, cat)
    images = glob.glob(os.path.join(cat_dir, "*.*"))
    for img in images:
        all_images.append({"category": cat, "image_path": img})

print(f"Total images found: {len(all_images)}")

# 2. Load recipe_dataset_2m.csv
print(f"Loading {RECIPE_CSV}...")
recipes_df = pd.read_csv(RECIPE_CSV)

# Parse ingredients if they are stored as stringified lists
if isinstance(recipes_df['ingredients'].iloc[0], str):
    try:
        recipes_df['ingredients'] = recipes_df['ingredients'].apply(ast.literal_eval)
    except Exception:
        pass

# The 2M recipe dataset uses 'directions' instead of 'instructions' and lacks an 'id' column
recipes_df = recipes_df.dropna(subset=['title', 'ingredients', 'directions'])
recipes_df = recipes_df.reset_index(names=['id']) # Add a synthetic 'id' incrementally
recipes_df['title'] = recipes_df['title'].str.lower()

print(f"Total recipes loaded: {len(recipes_df)}")
print(recipes_df.head(3))

Total categories found: 561
Total images found: 500726
Loading ../../data/datasets/recipe_dataset_2m.csv...
Total recipes loaded: 2231141
   id                  title  \
0   0    no-bake nut cookies   
1   1  jewell ball's chicken   
2   2            creamy corn   

                                         ingredients  \
0  [1 c. firmly packed brown sugar, 1/2 c. evapor...   
1  [1 small jar chipped beef, cut up, 4 boned chi...   
2  [2 (16 oz.) pkg. frozen corn, 1 (8 oz.) pkg. c...   

                                          directions  \
0  ["In a heavy 2-quart saucepan, mix brown sugar...   
1  ["Place chipped beef on bottom of baking dish....   
2  ["In a slow cooker, combine all ingredients. C...   

                                              link    source  \
0   www.cookbooks.com/Recipe-Details.aspx?id=44874  Gathered   
1  www.cookbooks.com/Recipe-Details.aspx?id=699419  Gathered   
2   www.cookbooks.com/Recipe-Details.aspx?id=10570  Gathered   

                          

# Task 1.2 Recipe Matching & Formatting

In [3]:
import concurrent.futures

# Sanitize directory/category names
category_to_keywords = {}
for cat in categories:
    sanitized = cat.lower().replace("_", " ")
    category_to_keywords[cat] = [sanitized]

    # You can add custom overrides here if needed
    if "macaron" in sanitized:
        category_to_keywords[cat].append("macaroon")

print("Sample category mappings:")
print(list(category_to_keywords.items())[:5])

# Convert lists to strings for fast vectorized regex matching
print("Preparing 'ingredients_str' for regex matching...")
if not isinstance(recipes_df['ingredients'].iloc[0], str):
    recipes_df['ingredients_str'] = recipes_df['ingredients'].apply(lambda x: " ".join(x) if isinstance(x, list) else str(x))
else:
    recipes_df['ingredients_str'] = recipes_df['ingredients']
    
def get_pool_matches_for_category(cat):
    keywords = category_to_keywords[cat]
    pattern = '|'.join(keywords)
    
    # Vectorized match
    title_match = recipes_df['title'].str.contains(pattern, case=False, na=False, regex=True)
    ing_match = recipes_df['ingredients_str'].str.contains(pattern, case=False, na=False, regex=True)
    
    matches = recipes_df[title_match | ing_match]
    
    # Return all matches (removed the 500 limit)
    if len(matches) > 0:
        return cat, matches
    return cat, pd.DataFrame()

paired_dataset = []
recipe_usage_count = {} # Use dict instead of set to allow reuse tracking
skipped = []

# Group images by category
images_by_cat = {}
for img_data in all_images:
    cat = img_data['category']
    if cat not in images_by_cat:
        images_by_cat[cat] = []
    images_by_cat[cat].append(img_data)

print('Pre-filtering recipe pools via regex and ThreadPoolExecutor...')
category_pools = {}
categories_list = list(images_by_cat.keys())

with concurrent.futures.ThreadPoolExecutor() as executor:
    results = list(tqdm(executor.map(get_pool_matches_for_category, categories_list), total=len(categories_list), desc="Regex Matching"))
    for cat, pool in results:
        category_pools[cat] = pool

print('Pairing images with recipes incrementally and tracking usage...')
for cat, images in tqdm(images_by_cat.items(), desc="Pairing"):
    sampled_imgs = images # Use all images instead of random sampling
    cat_matches = category_pools.get(cat, pd.DataFrame())
    
    if len(cat_matches) == 0:
        skipped.append(cat)
        continue
        
    for img_data in sampled_imgs:
        num_matches = random.randint(5, 10)
        
        # Deduplicate: only exclude recipes used 5 or more times
        overused_ids = [rid for rid, count in recipe_usage_count.items() if count >= 5]
        pool = cat_matches[~cat_matches['id'].isin(overused_ids)]
        
        if len(pool) == 0:
            break
        
        matches = pool.sample(min(len(pool), num_matches))
        for _, recipe in matches.iterrows():
            rid = recipe['id']
            recipe_usage_count[rid] = recipe_usage_count.get(rid, 0) + 1
            
            # Format text: Title, Ingredients, Instructions
            instructions = str(recipe.get('directions', ''))
            ingredients = ", ".join(recipe['ingredients']) if isinstance(recipe['ingredients'], list) else str(recipe['ingredients'])
            full_text = f"Title: {recipe['title']}\nIngredients: {ingredients}\nInstructions: {instructions}"
            
            paired_dataset.append({
                'image_path': img_data['image_path'],
                'category': cat,
                'recipe_title': recipe['title'],
                'ingredients': ingredients,
                'instructions': instructions,
                'full_text': full_text,
                'recipe_id': int(rid)
            })

print(f'\nTotal pairs built: {len(paired_dataset)}')
print(f'Skipped categories: {skipped}')

Sample category mappings:
[('Pork_tenderloin_sandwich', ['pork tenderloin sandwich']), ('Sausage_sandwich', ['sausage sandwich']), ('breakfast_burrito', ['breakfast burrito']), ('Goulash', ['goulash']), ('Fish_head_casserole', ['fish head casserole'])]
Preparing 'ingredients_str' for regex matching...
Pre-filtering recipe pools via regex and ThreadPoolExecutor...


Regex Matching:  19%|█▉        | 106/561 [00:03<00:10, 44.64it/s]/tmp/ipykernel_1473205/3544855909.py:28: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  title_match = recipes_df['title'].str.contains(pattern, case=False, na=False, regex=True)
/tmp/ipykernel_1473205/3544855909.py:29: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  ing_match = recipes_df['ingredients_str'].str.contains(pattern, case=False, na=False, regex=True)
Regex Matching: 100%|██████████| 561/561 [00:13<00:00, 41.25it/s]


Pairing images with recipes incrementally and tracking usage...


Pairing: 100%|██████████| 561/561 [44:29<00:00,  4.76s/it]


Total pairs built: 1289959
Skipped categories: ['Fish_head_casserole', 'Cobbler_food', 'Es_pisang_ijo', 'Taro_dumpling', 'Hae_mee', 'Fuqi_feipian', 'Soused_herring', 'Arròs_negre', 'Pasteis_de_Bacalhau', 'Cheese_maki', 'Huangqiao_sesame_cake', 'Gyutan', 'Mutton_handi', "Pig's_organ_soup", 'Crab_in_Padang_sauce', 'Bánh_canh', 'Shuizhu', 'Suanmeitang', 'Tinga_dish', 'Sliced_fish_soup', 'Stir-fried_tomato_and_scrambled_eggs', 'White_boiled_shrimp', 'Sopa_de_Mandioca', 'Yuxiangrousi', 'Gołąbki', 'Sapo_tahu', 'Mì_Quảng', 'Pork_Knuckles_and_Ginger_Stew', 'Bacon_egg_and_cheese_sandwich', 'Babi_panggang', 'Nattō', 'Hongshao_rou', 'Jiuniang', "Sheep's_trotters", 'Æbleflæsk', 'Svíčková', 'Jambonneau', 'Bánh_chuối', 'Steamed_meatball', 'Bún_riêu', 'Bisque_food', 'Wotou', 'Pizza_pugliese', 'Andong_jjimdak', 'Pork_blood_soup', 'Zongzi', 'Bún_ốc', 'Wenchang_chicken', 'Kwetiau_goreng', 'Butadon', 'Çiğ_köfte', 'Es_campur', 'Avocado_key_lime_pie', 'Botan-ebi', 'Mission_burrito', 'Gejang', 'Doufunao'

# Task 1.2.1 & 1.2.2 SigLIP Image-Recipe Score and VLM Validation

In [1]:
import torch
from PIL import Image
from transformers import AutoProcessor, AutoModel
import pandas as pd
from tqdm import tqdm
import base64
import requests
import json
import os

print("Loading SigLIP model...")
model = AutoModel.from_pretrained("google/siglip-base-patch16-224")
processor = AutoProcessor.from_pretrained("google/siglip-base-patch16-224")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

SCORED_FILE = "../../data/datasets/scored_dataset_tmp.json"

if os.path.exists(SCORED_FILE):
    with open(SCORED_FILE, "r") as f:
        scored_dataset = json.load(f)
    print(f"Resuming with {len(scored_dataset)} already scored pairs.")
else:
    scored_dataset = []

# Use a specific key (image path + recipe id) to deduplicate and identify already processed rows
scored_keys = {p['image_path'] + "||" + str(p['recipe_id']) for p in scored_dataset}

print("Scoring pairs with SigLIP...")
new_count = 0
for pair in tqdm(paired_dataset, desc="SigLIP Scoring"):
    pair_key = pair['image_path'] + "||" + str(pair['recipe_id'])
    
    # Checkpoint Guard: Skip if already scored
    if pair_key in scored_keys:
        continue
        
    title = pair['recipe_title']
    ingredients = pair['ingredients']  
    # Guardrail: Hard clip ingredients string and total text to avoid any buffer overflow 
    short_text = f"Title: {title} Ingredients: {ingredients}"[:300]
    image_path = pair['image_path']
    
    try:
        with Image.open(image_path) as img:
            image = img.convert('RGB')
            
        # Guardrail: Added `truncation=True`
        inputs = processor(text=[short_text], images=image, padding="max_length", truncation=True, return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        with torch.no_grad():
            outputs = model(**inputs)
            logits_per_image = outputs.logits_per_image
            probs = torch.sigmoid(logits_per_image) # convert logits to probability score
            score = probs[0][0].item()
            
        pair['siglip_score'] = score
        scored_dataset.append(pair)
        scored_keys.add(pair_key)
        new_count += 1
        
        # Checkpoint Guard: Save to disk every 5000 records so progress isn't lost on crash
        if new_count % 5000 == 0:
            with open(SCORED_FILE, "w") as f:
                json.dump(scored_dataset, f)
                
    except Exception as e:
        print(f"Error processing {image_path}: {e}")

# Final save after SigLIP loop
with open(SCORED_FILE, "w") as f:
    json.dump(scored_dataset, f)

# Threshold logic
df_scored = pd.DataFrame(scored_dataset)

# VLM Validation Checkpoint Setup
VLM_FILE = "../../data/datasets/vlm_processed_tmp.json"
if os.path.exists(VLM_FILE):
    with open(VLM_FILE, "r") as f:
        vlm_processed = json.load(f)
    print(f"Resuming VLM with {len(vlm_processed)} already evaluated pairs.")
else:
    vlm_processed = []

vlm_keys = {p['image_path'] + "||" + str(p['recipe_id']) for p in vlm_processed}
vlm_status_lookup = {p['image_path'] + "||" + str(p['recipe_id']): p.get('vlm_status') for p in vlm_processed}

def evaluate_with_vllm(image_path, title, ingredients, retries=3):
    try:
        with open(image_path, "rb") as f:
            encoded_string = base64.b64encode(f.read()).decode('utf-8')

        ing_text = ', '.join(ingredients[:8]) if isinstance(ingredients, list) else ingredients[:120]
        prompt = f"Image shows '{title}' made with: {ing_text}. Does this image match? Reply only: yes or no."

        payload = {
            "model": "qwen2.5vl:7b",
            "prompt": prompt,
            "images": [encoded_string],
            "stream": False,
            "options": {
                "temperature": 0,
                "num_predict": 4
            }
        }

        for attempt in range(retries):
            response = requests.post("http://localhost:11434/api/generate", json=payload)
            if response.status_code == 200:
                result = response.json().get("response", "").strip().lower()
                if result.startswith("yes"):
                    return True
                if result.startswith("no"):
                    return False
                payload["options"]["temperature"] = 0.1 * (attempt + 1)
            else:
                print(f"VLM HTTP error {response.status_code} on attempt {attempt + 1}")

        print(f"VLM gave no clear yes/no after {retries} attempts for {image_path}, defaulting to False")
        return False

    except Exception as e:
        print(f"VLM error {image_path}: {e}")
        return False

vllm_stats = {'accepted': 0, 'rejected': 0, 'auto_accepted': 0}
new_vlm_count = 0
high_quality_pairs = []

for cat, group in df_scored.groupby('category'):
    if group.empty: continue
    
    for _, row in tqdm(group.iterrows(), total=len(group), desc=f"VLM Validation ({cat})", leave=False):
        pair_dict = row.to_dict()
        pair_key = pair_dict['image_path'] + "||" + str(pair_dict['recipe_id'])
        
        # Checkpoint Guard: Reconstruct from file if already processed
        if pair_key in vlm_keys:
            status = vlm_status_lookup.get(pair_key)
            if status == 'auto_accepted': 
                vllm_stats['auto_accepted'] += 1
                high_quality_pairs.append(pair_dict)
            elif status == 'accepted': 
                vllm_stats['accepted'] += 1
                high_quality_pairs.append(pair_dict)
            elif status == 'rejected': 
                vllm_stats['rejected'] += 1
            continue
            
        score = row['siglip_score']
        
        # Global minimum threshold (0.3)
        if score < 0.3:
            pair_dict['vlm_status'] = 'rejected_by_threshold'
        elif score > 0.6:
            pair_dict['vlm_status'] = 'auto_accepted'
            vllm_stats['auto_accepted'] += 1
            high_quality_pairs.append(pair_dict)
        else:
            # Uncertain range: VLM validation via Ollama
            vllm_pass = evaluate_with_vllm(row['image_path'], row['recipe_title'], row['ingredients'])
            
            if vllm_pass:
                pair_dict['vlm_status'] = 'accepted'
                vllm_stats['accepted'] += 1
                high_quality_pairs.append(pair_dict)
            else:
                pair_dict['vlm_status'] = 'rejected'
                vllm_stats['rejected'] += 1
                
        vlm_processed.append(pair_dict)
        vlm_keys.add(pair_key)
        vlm_status_lookup[pair_key] = pair_dict['vlm_status']
        new_vlm_count += 1
        
        # Checkpoint Guard: Save VLM evaluations incrementally
        if new_vlm_count % 1000 == 0:
            with open(VLM_FILE, "w") as f:
                json.dump(vlm_processed, f)

# Final save after VLM loop
with open(VLM_FILE, "w") as f:
    json.dump(vlm_processed, f)

print(f"SigLIP + VLM Filtering Complete: {len(high_quality_pairs)} pairs retained.")
print(f"VL-LLM stats: {vllm_stats}")

/home/s22imc10262/data/NLP/hackathon/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading SigLIP model...


Loading weights: 100%|██████████| 408/408 [00:00<00:00, 7129.66it/s]


Resuming with 1289959 already scored pairs.
Scoring pairs with SigLIP...


NameError: name 'paired_dataset' is not defined

In [3]:
import json
import pandas as pd

VLM_FILE = "../../data/datasets/vlm_processed_tmp.json"

# Load the results directly from the temporary file
with open(VLM_FILE, "r") as f:
    vlm_processed = json.load(f)

# Reconstruct high_quality_pairs from the saved statuses
high_quality_pairs = [
    p for p in vlm_processed 
    if p.get('vlm_status') in ['auto_accepted', 'accepted']
]

print(f"Loaded {len(vlm_processed)} total processed pairs from the temp file.")
print(f"Recovered {len(high_quality_pairs)} high-quality pairs.")

Loaded 1289959 total processed pairs from the temp file.
Recovered 74281 high-quality pairs.


# Task 1.3 Exporting & Reporting and 1.4 Metadata Lookup System

In [6]:
import collections

# Task 1.3 Exporting & Reporting
df_high_quality = pd.DataFrame(high_quality_pairs)

if not df_high_quality.empty:
    count_by_cat = df_high_quality['category'].value_counts()
    print("Categories with < 20 pairs:")
    print(count_by_cat[count_by_cat < 20])
else:
    print("WARNING: No high quality pairs.")

final_paired = []
for p in high_quality_pairs:
    final_paired.append({
        'image_path': p['image_path'],
        'category': p['category'],
        'recipe_id': p['recipe_id'],
        'recipe_title': p['recipe_title'],
        'ingredients': p['ingredients'],
    })

paired_dataset_path = "../../data/datasets/paired_dataset3.json"
with open(paired_dataset_path, "w") as f:
    json.dump(final_paired, f, indent=2)

print(f"Saved {len(final_paired)} pairs to {paired_dataset_path}")

# Task 1.4 Metadata Lookup System (recipe_metadata.json)
recipe_metadata = {}
unique_ids_in_dataset = set(p['recipe_id'] for p in final_paired)

# Retrieve full text for the unique recipes used, or all of them depending on need
# Here we'll map all recipes_df for full metadata, or just those in pairs. But requirements say:
# "recipe_metadata.json file mapping recipe_id → {title, ingredients, instructions} (full recipe text)."
# For production we might want to store all 2M recipes, but here let's just do a subset or a demo.
# Note: full 2M recipes will result in a huge JSON. Usually we'd index all:
print("Creating recipe_metadata.json...")
metadata_dict = {}
for _, row in tqdm(recipes_df.iterrows(), total=len(recipes_df)):
    metadata_dict[str(row['id'])] = {
        'title': row['title'],
        'ingredients': row['ingredients'],
        'instructions': str(row.get('directions', ''))
    }

with open("../../data/indexes/recipe_index_metadata_newdata3.json", "w") as f:
    json.dump(metadata_dict, f)

print(f"Saved metadata for {len(metadata_dict)} recipes.")

Categories with < 20 pairs:
category
Bistek            19
Carpaccio         19
Chilli_crab       19
Ham_sandwich      19
Sambar            19
                  ..
Red_bean_cake      1
Tobiko             1
Tonkotsu_ramen     1
Torta_caprese      1
Wonton_noodles     1
Name: count, Length: 152, dtype: int64
Saved 74281 pairs to ../../data/datasets/paired_dataset3.json
Creating recipe_metadata.json...


100%|██████████| 2231141/2231141 [00:47<00:00, 46845.46it/s]


Saved metadata for 2231141 recipes.
